# Deep Hedging AI (JAX port)
### Recurrent Network: impact of using recurrent networks on hedging forward started options

Port of `notebooks/trainer-recurrent-fwdstart.ipynb` by Hans Buehler, January 2023.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import math
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display, Markdown
from deephedging.world import WorldConfig, simulate
from deephedging.model import AgentConfig, GymConfig
from deephedging.train import TrainConfig, fit
from deephedging.plotting import plot_training
from deephedging.stats import mean_bins

# DH_EPOCHS overrides every epoch count for quick smoke runs; DH_CHECKPOINTS="" turns checkpoints off.
epochs = lambda n: int(os.environ.get("DH_EPOCHS") or n)
CHECKPOINTS = os.environ.get("DH_CHECKPOINTS", "checkpoints") or None

In [ ]:
iReset = 10
nBins = 200

def fwd_start(spots):
    terminal = np.maximum(1. * spots[:, iReset] - spots[:, -1], 0.)  # put
    return -terminal

world_cfg = WorldConfig(steps=20, samples=20000, drift=0., cost_s=0., black_scholes=True, payoff=fwd_start)
agent = AgentConfig(depth=5, activation="softplus")
gym = GymConfig(agent=agent, utility="cvar", lmbda=1.01)
train_cfg = TrainConfig(epochs=epochs(2000), batch_size=32, learning_rate=0.001, clipvalue=0.9, global_clipnorm=1.)

world = simulate(world_cfg)
val_world = world.clone(samples=world.nSamples // 2, seed=1)
fits = {}

# plot
payoff = world.market["payoff"]
spots = world.spot_all
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

ref = spots[:, -1] / spots[:, iReset]
ixs = np.argsort(ref)
xm = mean_bins(ref[ixs], bins=nBins, weights=None, return_std=False)
m, s = mean_bins(payoff[ixs], bins=nBins, weights=None, return_std=True)
axs[0].plot(xm, m, color="blue")
axs[0].set_title("Payoff (over forward return)")
axs[0].set_xlabel("Spot/Reset")
axs[0].fill_between(xm, m - s, m + s, color="blue", alpha=0.2)

ref = spots[:, -1]
ixs = np.argsort(ref)
xm = mean_bins(ref[ixs], bins=nBins, weights=None, return_std=False)
m, s = mean_bins(payoff[ixs], bins=nBins, weights=None, return_std=True)
axs[1].plot(xm, m, color="blue")
axs[1].set_title("Payoff (over terminal return)")
axs[1].set_xlabel("Spot")
axs[1].fill_between(xm, m - s, m + s, color="blue", alpha=0.2)
plt.show()

In [ ]:
display(Markdown("## Hedging a forward started put in non-recurrent mode"))
fits["none"] = fit(replace(gym, agent=replace(agent, classic=0, aggregate=0, past_repr=0, event=0)), train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(fits["none"].result, fits["none"].train, fits["none"].val, world, val_world)
plt.show()

In [ ]:
display(Markdown("## Hedging a forward started put with all node modes"))
fits["all_modes"] = fit(replace(gym, agent=replace(agent, classic=2, aggregate=2, past_repr=2, event=2)), train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(fits["all_modes"].result, fits["all_modes"].train, fits["all_modes"].val, world, val_world)
plt.show()

In [ ]:
display(Markdown("## Hedging a forward started put with 2 'aggregate' nodes only"))
fits["aggregate_2"] = fit(replace(gym, agent=replace(agent, classic=0, aggregate=2, past_repr=0, event=0)), train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(fits["aggregate_2"].result, fits["aggregate_2"].train, fits["aggregate_2"].val, world, val_world)
plt.show()

In [ ]:
display(Markdown("## Hedging a forward started put with 5 'bound aggregate' nodes only"))
# the original cell assigns 'event' twice and never resets 'past_repr', so this variant keeps past_repr=1
fits["aggregate_5bnd"] = fit(replace(gym, agent=replace(agent, classic=0, aggregate=5, past_repr=1, event=0, bound_aggr_states=True)), train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(fits["aggregate_5bnd"].result, fits["aggregate_5bnd"].train, fits["aggregate_5bnd"].val, world, val_world)
plt.show()

In [ ]:
display(Markdown("## Hedging a forward started put with 5 unbound 'aggregate' nodes only"))
fits["aggregate_5"] = fit(replace(gym, agent=replace(agent, classic=0, aggregate=5, past_repr=0, event=0)), train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(fits["aggregate_5"].result, fits["aggregate_5"].train, fits["aggregate_5"].val, world, val_world)
plt.show()

In [ ]:
display(Markdown("## Hedging a forward started put with 'past_repr' nodes only"))
fits["past_repr"] = fit(replace(gym, agent=replace(agent, classic=0, aggregate=0, past_repr=2, event=0)), train_cfg, world, val_world, log_every=100, checkpoint_dir=CHECKPOINTS)
plot_training(fits["past_repr"].result, fits["past_repr"].train, fits["past_repr"].val, world, val_world)
plt.show()

In [ ]:
spots = world.spot_all
hedges = world.market["hedges"][:, :, 0]
time_left = world.features["time_left"][0, :, 0]
nSteps = world.nSteps
nSamples = world.nSamples
nBins = 200

actions = {k: fits[k].train["actions"][:, :, 0] for k in fits}
deltas = {k: np.cumsum(actions[k], axis=1) for k in fits}

vol = world_cfg.rvol

pnls = {k: np.zeros((nSamples,)) for k in fits}
pnls["blackscholes"] = np.zeros((nSamples,))
prev_deltas_t = {k: 0. for k in pnls}

keys_sorted = ["blackscholes"] + list(fits.keys())
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_of = lambda i: color_cycle[i % len(color_cycle)]

display(Markdown("## Forward Starting Option Hedging"))
ncols = 5
fig, axs = plt.subplots(math.ceil(nSteps / ncols), ncols, figsize=(4 * ncols, 3.5 * math.ceil(nSteps / ncols)))
axs = axs.ravel()

for j in range(nSteps):
    ax_delta = axs[j]
    ax_delta.set_ylim(-1.1, 0.1)

    # sort by spot at j, and compute BS reference
    spot_t = spots[:, j]
    hedges_t = hedges[:, j]   # S(T)-S(t_j)
    res_t = time_left[j]

    # BS
    # note that the 'drift' in the simulator is the statistical drift, not the risk-neutral drift.
    if j < iReset:
        ref_spot = spot_t
        bsdelta_t = ref_spot * 0.
        ax_delta.set_title("Delta step %ld (before reset)" % j)
        ax_delta.set_xlabel("Spot")
    else:
        strike = spots[:, iReset]
        d1 = (np.log(spot_t / strike) + 0.5 * vol * vol * res_t) / math.sqrt(res_t * vol * vol)
        d2 = d1 - vol * math.sqrt(res_t)
        N1 = norm.cdf(d1)
        N2 = norm.cdf(d2)
        bsprice_t = spot_t * N1 - strike * N2      # call
        bsprice_t = bsprice_t + strike - spot_t    # C=P+S-K
        bsdelta_t = N1
        bsdelta_t = bsdelta_t - 1.
        ref_spot = spot_t / strike
        ax_delta.set_title("Delta step %ld (after reset)" % j)
        ax_delta.set_xlabel("Spot/Strike")

    deltas_t = {k: deltas[k][:, j] for k in deltas}
    deltas_t["blackscholes"] = bsdelta_t
    actions_t = {k: deltas_t[k] - prev_deltas_t[k] for k in deltas_t}
    dS_t = spots[:, j + 1] - spot_t
    pnls = {k: pnls[k] + actions_t[k] * dS_t for k in pnls}

    # plot
    ixs = np.argsort(ref_spot)
    ref_spot = ref_spot[ixs]
    deltas_t = {k: deltas_t[k][ixs] for k in deltas_t}

    x_mean = mean_bins(ref_spot, bins=nBins, weights=None, return_std=False)
    for i, k in enumerate(keys_sorted):
        mean, std = mean_bins(deltas_t[k], bins=nBins, weights=None, return_std=True)
        ax_delta.plot(x_mean, mean, label=k, color=color_of(i))
        ax_delta.fill_between(x_mean, mean - std, mean + std, color=color_of(i), alpha=0.1)

    ax_delta.legend()

fig.tight_layout()
plt.show()

# terminal
display(Markdown("## Terminal Profiles"))

spots = world.spot_all
payoff = world.market["payoff"]
payoff = payoff - np.mean(payoff)

spot_ret = spots[:, -1] / spots[:, iReset]
ixs = np.argsort(spot_ret)
spot_ret = spot_ret[ixs]
pnls = {k: pnls[k][ixs] for k in pnls}
payoff = payoff[ixs]

spot_mean = mean_bins(spot_ret, bins=nBins, weights=None, return_std=False)
payoff_mean, payoff_std = mean_bins(payoff, bins=nBins, weights=None, return_std=True)

fig, ax = plt.subplots(figsize=(16, 12))
ax.set_title("Terminal")
ax.set_xlabel("Spot/Strike")

ax.plot(spot_mean, -payoff_mean, label="-payoff", color=color_of(0))
ax.fill_between(spot_mean, -payoff_mean - payoff_std, -payoff_mean + payoff_std, color=color_of(0), alpha=0.1)
for i, k in enumerate(keys_sorted):
    mean, std = mean_bins(pnls[k], bins=nBins, weights=None, return_std=True)
    ax.plot(spot_mean, mean, label=k, color=color_of(i + 1))
    ax.fill_between(spot_mean, mean - std, mean + std, color=color_of(i + 1), alpha=0.1)

ax.legend()
plt.show()